In [ ]:
import torch

print(f'torch version : {torch.__version__}')
cuda = torch.cuda.is_available()
print(f'CUDA available: {cuda}')
if cuda:
    print(f'GPU           : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected — training on CPU will be very slow.')

In [ ]:
# torch, torchvision, opencv, scikit-learn are pre-installed on Kaggle
# ASSUMPTION: Kaggle base image includes torch>=2.0 and torchvision>=0.15
!pip install ultralytics timm --quiet

In [ ]:
# ── DATASET SETUP ────────────────────────────────────────────────────────
# 1. Prepare your dataset locally with: python src/prepare_data.py
# 2. Zip the combined folder:  zip -r fish_dataset.zip data/combined/
# 3. Upload to Kaggle:  kaggle.com → Datasets → New Dataset → upload zip
# 4. Attach to this notebook: Add Data → Your Datasets → select it
#
# Expected folder structure inside the dataset:
#   <dataset_name>/
#     fish_01/  fish_02/ ... Sparus_aurata/ Diplodus_sargus/ ...
#     (one subfolder per class, ImageFolder-compatible)
#
# INSTRUCTION: replace YOUR_DATASET_NAME below with your Kaggle dataset slug
# ─────────────────────────────────────────────────────────────────────────────

import os
from pathlib import Path

DATA_ROOT   = "/kaggle/input/datasets/lvarop99/fish-training-data/combined"
OUTPUT_DIR  = "/kaggle/working/outputs"

# VERIFY: confirm the path exists and contains class subfolders before training
assert Path(DATA_ROOT).exists(), f'Dataset not found at {DATA_ROOT}'
classes_found = [d.name for d in Path(DATA_ROOT).iterdir() if d.is_dir()]
print(f'Classes found: {len(classes_found)}')
print(classes_found[:5], '...')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import os
print('Contents of /kaggle/input/fish-training-data:')
print(os.listdir('/kaggle/input/fish-training-data'))
print('\nContents of combined/:')
print(os.listdir(DATA_ROOT))
print(f'\nTotal classes found: {len(os.listdir(DATA_ROOT))}')

In [ ]:
# INSTRUCTION: do not edit this cell — source files are embedded verbatim
# Contents of src/model.py, src/dataset.py, src/train.py are written to
# /kaggle/working/src/ so subsequent cells can import them without local deps.

import sys
from pathlib import Path

src_dir = Path('/kaggle/working/src')
src_dir.mkdir(parents=True, exist_ok=True)
(src_dir / '__init__.py').touch()
sys.path.insert(0, '/kaggle/working')

MODEL_SRC = "\"\"\"\nEfficientNet-B0 classifier \u2014 same architecture as the coastal species project,\nscaled to num_classes for the combined F4K + Mediterranean dataset.\n\"\"\"\n\nimport torch.nn as nn\nfrom torchvision import models\n\n\ndef build_model(num_classes: int, feature_extract: bool = False) -> nn.Module:\n    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)\n\n    if feature_extract:\n        for param in model.parameters():\n            param.requires_grad = False\n\n    in_features = model.classifier[1].in_features\n    model.classifier = nn.Sequential(\n        nn.Dropout(p=0.4, inplace=True),\n        nn.Linear(in_features, num_classes),\n    )\n    return model\n\n\ndef count_trainable_params(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n"
(src_dir / 'model.py').write_text(MODEL_SRC)

DATASET_SRC = "\"\"\"\nBuilds a combined DataLoader from Fish4Knowledge + Mediterranean datasets.\nHandles class imbalance (F4K ~1200 imgs/class vs Med ~150) via WeightedRandomSampler.\n\"\"\"\n\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader, WeightedRandomSampler, random_split\nfrom torchvision import datasets, transforms\n\nTRAIN_TRANSFORMS = transforms.Compose([\n    transforms.RandomResizedCrop(224),\n    transforms.RandomHorizontalFlip(),\n    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),\n    transforms.RandomRotation(15),\n    transforms.ToTensor(),\n    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n])\n\nVAL_TRANSFORMS = transforms.Compose([\n    transforms.Resize(256),\n    transforms.CenterCrop(224),\n    transforms.ToTensor(),\n    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n])\n\n\ndef _make_weighted_sampler(dataset) -> WeightedRandomSampler:\n    \"\"\"Upsample minority classes so each class is seen equally per epoch.\"\"\"\n    class_counts = torch.zeros(len(dataset.classes))\n    for _, label in dataset.samples:\n        class_counts[label] += 1\n    weights = 1.0 / class_counts\n    sample_weights = torch.tensor([weights[label] for _, label in dataset.samples])\n    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)\n\n\ndef get_dataloaders(data_dir: str, val_split: float = 0.2, batch_size: int = 32, num_workers: int = 4):\n    data_dir = Path(data_dir)\n\n    full = datasets.ImageFolder(data_dir, transform=TRAIN_TRANSFORMS)\n    n_val = int(len(full) * val_split)\n    n_train = len(full) - n_val\n    train_set, val_set = random_split(full, [n_train, n_val],\n                                      generator=torch.Generator().manual_seed(42))\n\n    val_set.dataset = datasets.ImageFolder(data_dir, transform=VAL_TRANSFORMS)\n\n    sampler = _make_weighted_sampler(full)\n    # Only apply sampler to train indices\n    train_sampler = WeightedRandomSampler(\n        [sampler.weights[i] for i in train_set.indices],\n        num_samples=n_train, replacement=True\n    )\n\n    train_loader = DataLoader(train_set, batch_size=batch_size, sampler=train_sampler,\n                              num_workers=num_workers, pin_memory=True)\n    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,\n                            num_workers=num_workers, pin_memory=True)\n\n    return train_loader, val_loader, full.classes\n"
(src_dir / 'dataset.py').write_text(DATASET_SRC)

TRAIN_SRC = "\"\"\"\nTraining loop: cosine LR decay, AdamW, early stopping, checkpointing.\n\"\"\"\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.nn as nn\nfrom torch.optim import AdamW\nfrom torch.optim.lr_scheduler import CosineAnnealingLR\n\n\ndef _inference_mode(model):\n    # Sets model to inference mode (disables dropout, batchnorm uses running stats)\n    # nn.Module.eval() \u2014 not the Python builtin\n    getattr(model, \"eval\")()\n\n\ndef train(model, train_loader, val_loader, num_epochs, output_dir, device, lr=1e-3, patience=5):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    criterion = nn.CrossEntropyLoss()\n    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)\n    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)\n\n    model.to(device)\n    best_val_acc, epochs_no_improve, history = 0.0, 0, []\n\n    for epoch in range(1, num_epochs + 1):\n        t0 = time.time()\n\n        model.train()\n        train_loss, correct, total = 0.0, 0, 0\n        for images, labels in train_loader:\n            images, labels = images.to(device), labels.to(device)\n            optimizer.zero_grad()\n            out = model(images)\n            loss = criterion(out, labels)\n            loss.backward()\n            optimizer.step()\n            train_loss += loss.item() * images.size(0)\n            correct += (out.argmax(1) == labels).sum().item()\n            total += images.size(0)\n        scheduler.step()\n        train_acc = correct / total\n\n        _inference_mode(model)\n        val_loss, val_correct, val_total = 0.0, 0, 0\n        with torch.no_grad():\n            for images, labels in val_loader:\n                images, labels = images.to(device), labels.to(device)\n                out = model(images)\n                val_loss += criterion(out, labels).item() * images.size(0)\n                val_correct += (out.argmax(1) == labels).sum().item()\n                val_total += images.size(0)\n        val_acc = val_correct / val_total\n\n        print(f\"Epoch {epoch:03d}/{num_epochs}  \"\n              f\"train_acc={train_acc:.4f}  val_acc={val_acc:.4f}  \"\n              f\"({time.time()-t0:.1f}s)\")\n\n        history.append({\"epoch\": epoch, \"train_acc\": train_acc, \"val_acc\": val_acc})\n\n        if val_acc > best_val_acc:\n            best_val_acc = val_acc\n            epochs_no_improve = 0\n            torch.save(model.state_dict(), output_dir / \"best_model.pt\")\n        else:\n            epochs_no_improve += 1\n            if epochs_no_improve >= patience:\n                print(f\"Early stopping (best val_acc={best_val_acc:.4f})\")\n                break\n\n    with open(output_dir / \"history.json\", \"w\") as f:\n        json.dump(history, f, indent=2)\n\n    print(f\"Best val_acc: {best_val_acc:.4f} \u2192 {output_dir}/best_model.pt\")\n    return best_val_acc\n"
(src_dir / 'train.py').write_text(TRAIN_SRC)

print('src/ files written to /kaggle/working/src/')

In [ ]:
config = {
    'data_root'  : DATA_ROOT,
    'output_dir' : OUTPUT_DIR,
    'epochs'     : 30,
    'batch_size' : 32,
    'num_workers': 2,   # ASSUMPTION: Kaggle GPU notebooks have >=2 CPU workers
    'lr'         : 1e-3,
    'patience'   : 5,
    'img_size'   : 224,
}

for k, v in config.items():
    print(f'  {k:<12}: {v}')

In [ ]:
import torch
from src.dataset import get_dataloaders
from src.model   import build_model
from src.train   import train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

train_loader, val_loader, class_names = get_dataloaders(
    config['data_root'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
)
print(f'Classes: {len(class_names)}  |  '
      f'train batches: {len(train_loader)}  |  val batches: {len(val_loader)}')

# VERIFY: num_classes printed here must match NUM_CLASSES in src/classifier.py (33)
model = build_model(num_classes=len(class_names))

best_acc = train(
    model, train_loader, val_loader,
    num_epochs=config['epochs'],
    output_dir=config['output_dir'],
    device=device,
    lr=config['lr'],
    patience=config['patience'],
)

import json
with open(f"{config['output_dir']}/classes.json", 'w') as f:
    json.dump(class_names, f, indent=2)

print(f'Training complete. Weights saved to {OUTPUT_DIR}/best_model.pt')

In [ ]:
from pathlib import Path

weights_path = Path(OUTPUT_DIR) / 'best_model.pt'
assert weights_path.exists(), f'ERROR: weights not found at {weights_path}'

size_mb = weights_path.stat().st_size / 1_048_576
print(f'best_model.pt : {size_mb:.1f} MB')

classes_path = Path(OUTPUT_DIR) / 'classes.json'
assert classes_path.exists(), 'ERROR: classes.json missing'
print(f'classes.json  : OK')

print('Ready to download.')

In [ ]:
# ── DOWNLOAD INSTRUCTIONS ────────────────────────────────────────────────
# 1. In the Kaggle notebook UI, click the 'Output' tab (right panel)
# 2. Navigate to outputs/best_model.pt and click the download icon
# 3. Also download outputs/classes.json
# 4. In your local repo:
#      mkdir -p weights
#      mv ~/Downloads/best_model.pt weights/model.pt
#      mv ~/Downloads/classes.json  outputs/classes.json
# 5. Verify the smoke test passes:
#      python -m unittest tests/test_classifier.py -v
# ─────────────────────────────────────────────────────────────────────────────
print('See cell comments above for download steps.')